In [7]:
import cobra
from corda import CORDA

import pandas as pd
import random

In [8]:
# load inputs
build_files_path = '/data2/hratch/human_me/build_files/'
full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
rmd = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)

In [20]:
def build_toy_model(full_model, rmd, max_score = 3, frac_reactions = 0.01):
    '''Generate small toy model from full Recon2.2'''

    # generate reaction confidence scores
    n_reactions = len(full_model.reactions)
    n_reactions_to_keep = round(frac_reactions*n_reactions)

    one_reaction = [r.id for r in full_model.reactions if len(r.genes)==1]
    reactions_to_exclude = random.sample(one_reaction, round(len(one_reaction)*.25)) # decreases prob of choosing a reaction with a gene by 4x
    reactions_to_exclude += [r.id for r in full_model.reactions if len(r.genes)>1]


    population = sorted(set([r.id for r in full_model.reactions]).difference(reactions_to_exclude))
    reactions_to_keep = random.sample(population, k = n_reactions_to_keep)

    conf = {}
    for r in full_model.reactions: 
        if r.id not in reactions_to_keep:
            conf[r.id] = -1
        else:
            conf[r.id] = random.choice(list(range(max_score +1)))
    conf["biomass_reaction"] = 3
    
    print('Extract model')
    opt = CORDA(full_model, conf, met_prod = rmd.index.tolist())
    opt.build()
    print(opt)
    
    print('Generate cobra model')
    toy_model = opt.cobra_model('toy_model')
    
    return toy_model

In [22]:
iter_, max_iter = 0, 10
first = True
min_growth = 1e-3
toy_model = cobra.Model('')
opt_val = toy_model.slim_optimize()

while (iter_ < max_iter) and opt_val < min_growth:
    print('iteration: {}'.format(iter_))
    
    toy_model = build_toy_model(full_model, rmd)
    opt_val = toy_model.slim_optimize()

    print('Growth value: {}'.format(opt_val))
    print('--------')
    iter_ += 1

iteration: 0
Extract model
build status: reconstruction complete
Inc. reactions: 615/7844
 - unclear: 1/27
 - exclude: 536/7706
 - low and medium: 5/33
 - high: 73/78

Generate cobra model
Growth value: 46.04088766923477
--------


In [23]:
print(opt_val)
if opt_val >= min_growth:
    cobra.io.save_json_model(model = toy_model, filename = '/data2/hratch/human_me/input_files/toy_model.json')
    

46.04088766923477
